# Views no Azure Databricks

Este documento resume os principais conceitos sobre Views no Azure Databricks, abordando os diferentes tipos de views e seus casos de uso.

---

## 1. O que é uma View?

Uma **View** é uma consulta lógica (query) armazenada contra uma ou mais tabelas de origem. Em Azure Databricks, uma view é equivalente a um Spark DataFrame persistido como um objeto em um schema.

**Características principais:**
- Não armazena dados fisicamente (apenas o texto da query)
- Executa a query toda vez que é consultada
- Pode ser acessada de qualquer lugar no Databricks (com as devidas permissões)
- Simplifica consultas complexas e promove reusabilidade

### Exemplo Visual

```
┌─────────────┐     ┌─────────────┐
│  table_1    │     │  table_2    │
├─────────────┤     ├─────────────┤
│ A1 A2 A3 A4 │     │ B1 B2 B3 B4 │
└──────┬──────┘     └──────┬──────┘
       │                   │
       └─────────┬─────────┘
                 │
                 ▼
         ┌─────────────┐
         │   view_x    │
         ├─────────────┤
         │ A1 A4 B2 B3 │
         └─────────────┘
```

```sql
CREATE VIEW view_x
AS SELECT A1, A4, B2, B3
FROM table_1
INNER JOIN table_2 ON ...
```

> 📘 **Da documentação Azure**: Uma view armazena o texto de uma query tipicamente contra uma ou mais fontes de dados ou tabelas no metastore. Criar uma view não processa ou escreve nenhum dado - apenas o texto da query é registrado no metastore no schema associado.

---

## 2. Tipos de Views

O Azure Databricks suporta três tipos principais de views:

### 2.1 Stored Views (Views Persistidas)

São objetos **persistentes** armazenados no catálogo/database. Permanecem disponíveis até serem explicitamente removidas.

```sql
CREATE VIEW view_name
AS query

-- Exemplo com comentário e colunas nomeadas
CREATE OR REPLACE VIEW experienced_employee 
  (id COMMENT 'Unique identification number', Name)
  COMMENT 'View for experienced employees'
AS SELECT id, name 
FROM all_employee 
WHERE working_years > 5;
```

**Características:**
- Persistidas no database/catalog
- Removidas apenas com `DROP VIEW`
- Visíveis para todos os usuários com permissão
- Requerem `USE CATALOG` e `USE SCHEMA` para acesso

---

### 2.2 Temporary Views (Views Temporárias)

São views com **escopo de sessão** - existem apenas durante a sessão Spark atual.

```sql
CREATE TEMP VIEW view_name
AS query

-- ou
CREATE TEMPORARY VIEW view_name
AS query

-- Exemplo
CREATE TEMPORARY VIEW subscribed_movies
AS SELECT mo.member_id, mb.full_name, mo.movie_title
FROM movies AS mo
INNER JOIN members AS mb ON mo.member_id = mb.id;
```

**Características:**
- Escopo de sessão (session-scoped)
- Removidas automaticamente quando a sessão termina
- Não são registradas no catálogo
- Ideais para cálculos intermediários

#### Criando via DataFrame API (Python)

```python
# Criar temporary view a partir de DataFrame
df.createOrReplaceTempView("temp_view_name")

# Consultar a view
spark.sql("SELECT * FROM temp_view_name").show()
```

---

### 2.3 Global Temporary Views

São views com **escopo de cluster** - disponíveis para todas as sessões no mesmo cluster.

```sql
CREATE GLOBAL TEMP VIEW view_name
AS query

-- Para consultar, usar o schema global_temp
SELECT * FROM global_temp.view_name
```

**Características:**
- Escopo de cluster (cluster-scoped)
- Removidas quando o cluster é reiniciado
- Armazenadas no schema especial `global_temp`
- Compartilhadas entre todas as sessões do cluster

> ⚠️ **Da documentação Azure**: Global temp views são um recurso legado do Azure Databricks, um resquício do Hive e HDFS. **O Databricks recomenda não usar global temp views** em novos projetos.

---

## 3. Spark Session - Eventos que Encerram a Sessão

Uma **Spark Session** é encerrada (e consequentemente as Temporary Views são perdidas) nos seguintes eventos:

| Evento | Descrição |
|--------|-----------|
| 🆕 Abrir novo notebook | Cada notebook inicia uma nova sessão |
| 🔄 Detach/Reattach cluster | Desconectar e reconectar ao cluster |
| 📦 Instalar pacote Python | Instalação de pacotes reinicia a sessão |
| ♻️ Reiniciar cluster | Reinício completo do cluster |

---

## 4. Comparativo entre Tipos de Views

| Aspecto | Stored Views | Temp Views | Global Temp Views |
|---------|--------------|------------|-------------------|
| **Persistência** | Persistida no DB | Sessão | Cluster |
| **Remoção** | `DROP VIEW` | Fim da sessão | Reinício do cluster |
| **Comando** | `CREATE VIEW` | `CREATE TEMP VIEW` | `CREATE GLOBAL TEMP VIEW` |
| **Acesso** | Qualquer sessão | Apenas sessão atual | Todas sessões do cluster |
| **Schema** | Database específico | Não registrada | `global_temp` |
| **Recomendado** | ✅ Sim | ✅ Sim | ⚠️ Legacy |

---

## 5. Tipos Adicionais de Views (Avançado)

Além dos três tipos básicos, o Azure Databricks oferece tipos avançados:

### 5.1 Materialized Views

Views que **pré-computam e armazenam** os resultados fisicamente. Ideais para queries frequentes e pesadas.

```sql
CREATE MATERIALIZED VIEW mv_sales
AS SELECT date, SUM(sales) AS total_sales
FROM transactions
GROUP BY date;

-- Atualizar a materialized view
REFRESH MATERIALIZED VIEW mv_sales;
```

**Características:**
- Armazenam resultados fisicamente (como Delta Tables)
- Atualização incremental quando possível
- Requerem Unity Catalog
- Melhor performance para queries complexas
---

## 6. Comandos Úteis

### Listar Views

```sql
-- Listar views no schema atual
SHOW VIEWS;

-- Listar views de um schema específico
SHOW VIEWS FROM schema_name;

-- Listar views com filtro
SHOW VIEWS LIKE 'sales*';

-- Ver detalhes de uma view
DESCRIBE VIEW view_name;
DESCRIBE EXTENDED view_name;
```

### Remover Views

```sql
-- Remover view persistida
DROP VIEW view_name;

-- Remover se existir (evita erro)
DROP VIEW IF EXISTS view_name;

-- Remover materialized view
DROP MATERIALIZED VIEW mv_name;
```

---

## 7. Diagrama: Ciclo de Vida das Views

```
┌─────────────────────────────────────────────────────────────────────┐
│                    Ciclo de Vida das Views                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌─────────────────┐                                               │
│  │  STORED VIEW    │ ──────────────────────────────────────┐       │
│  │  (Persistida)   │                                       │       │
│  └────────┬────────┘                                       │       │
│           │ CREATE VIEW                                    │       │
│           ▼                                                │       │
│  ┌─────────────────┐                                       │       │
│  │   Database/     │  Persiste até DROP VIEW               │       │
│  │   Catalog       │◄──────────────────────────────────────┘       │
│  └─────────────────┘                                               │
│                                                                     │
│  ┌─────────────────┐                                               │
│  │   TEMP VIEW     │ ──────────────────────────────────────┐       │
│  │ (Session-scoped)│                                       │       │
│  └────────┬────────┘                                       │       │
│           │ CREATE TEMP VIEW                               │       │
│           ▼                                                │       │
│  ┌─────────────────┐                                       │       │
│  │  SparkSession   │  Persiste até sessão encerrar         │       │
│  │    (atual)      │◄──────────────────────────────────────┘       │
│  └─────────────────┘                                               │
│                                                                     │
│  ┌─────────────────┐                                               │
│  │ GLOBAL TEMP VIEW│ ──────────────────────────────────────┐       │
│  │(Cluster-scoped) │                                       │       │
│  └────────┬────────┘                                       │       │
│           │ CREATE GLOBAL TEMP VIEW                        │       │
│           ▼                                                │       │
│  ┌─────────────────┐                                       │       │
│  │  global_temp    │  Persiste até cluster reiniciar       │       │
│  │    schema       │◄──────────────────────────────────────┘       │
│  └─────────────────┘                                               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

---

## 8. Boas Práticas

1. **Prefira Stored Views** para lógica compartilhada entre equipes
2. **Use Temp Views** para transformações intermediárias em notebooks
3. **Evite Global Temp Views** - são consideradas legacy
4. **Considere Materialized Views** para queries pesadas e frequentes
5. **Sempre referencie tabelas por nome**, não por path/URI
6. **Documente suas views** usando COMMENT

---

## Links para Consulta

### Documentação Oficial Microsoft/Azure Databricks

- [What is a view?](https://learn.microsoft.com/en-us/azure/databricks/views/) - Visão geral sobre views
- [CREATE VIEW](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-create-view) - Sintaxe completa do CREATE VIEW
- [DROP VIEW](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-drop-view) - Referência para remover views
- [SHOW VIEWS](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-aux-show-views) - Listar views disponíveis
- [Materialized Views](https://learn.microsoft.com/en-us/azure/databricks/ldp/materialized-views) - Guia de Materialized Views
- [Use materialized views in Databricks SQL](https://learn.microsoft.com/en-us/azure/databricks/ldp/dbsql/materialized) - Uso prático de MVs
- [CREATE MATERIALIZED VIEW](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-create-materialized-view) - Sintaxe de criação
- [ALTER MATERIALIZED VIEW](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-alter-materialized-view) - Alterar MVs
- [Configure materialized views](https://learn.microsoft.com/en-us/azure/databricks/ldp/dbsql/materialized-configure) - Configurações de MVs
- [CREATE TEMPORARY VIEW (pipelines)](https://learn.microsoft.com/en-us/azure/databricks/ldp/developer/ldp-sql-ref-create-temporary-view) - Views temporárias em pipelines

---

## Resumo Rápido

| Tipo | Comando | Escopo | Quando Usar |
|------|---------|--------|-------------|
| Stored View | `CREATE VIEW` | Permanente | Lógica compartilhada, produção |
| Temp View | `CREATE TEMP VIEW` | Sessão | Transformações intermediárias |
| Global Temp View | `CREATE GLOBAL TEMP VIEW` | Cluster | ⚠️ Evitar (legacy) |
| Materialized View | `CREATE MATERIALIZED VIEW` | Permanente + dados | Queries pesadas, dashboards |

---